In [ ]:
# Install Mazinger and start a local, OpenAI-compatible Homura server.
import os
import sys
import time
import json
import shutil
import zipfile
import subprocess
import urllib.request
from pathlib import Path

os.makedirs("/kaggle/working", exist_ok=True)
os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPENAI_MODEL"] = "IndexTeam/Index-Homura-9B"
os.environ["MAZINGER_DISABLE_VISION"] = "1"

archive_candidates = list(Path("/kaggle/input").rglob("mazinger_source.zip"))
if not archive_candidates:
    raise FileNotFoundError("mazinger_source.zip is missing from Kaggle input; re-upload the dataset with the root-level source archive.")
mazinger_extract_path = Path("/kaggle/working/mazinger")
if mazinger_extract_path.exists():
    shutil.rmtree(mazinger_extract_path)
mazinger_extract_path.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_candidates[0]) as source_archive:
    extraction_root = mazinger_extract_path.resolve()
    for member in source_archive.infolist():
        destination = (extraction_root / member.filename).resolve()
        if destination != extraction_root and extraction_root not in destination.parents:
            raise RuntimeError("Mazinger source archive contains an unsafe path.")
    source_archive.extractall(extraction_root)
mazinger_candidates = [
    path.parent for path in mazinger_extract_path.rglob("pyproject.toml")
    if path.parent.name.lower() == "mazinger" or path.parent == mazinger_extract_path
]
if not mazinger_candidates:
    raise FileNotFoundError("pyproject.toml was not found inside mazinger_source.zip.")
mazinger_path = str(mazinger_candidates[0])
print(f"Installing bundled Mazinger source: {mazinger_path}", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-cache-dir", f"{mazinger_path}[all]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "faster-whisper", "demucs", "omnivoice"], check=True)

# vLLM/model download can take a while on a fresh Kaggle session; allow up to 15 min.
# Kaggle's common T4 x2 setup needs tensor parallelism for Homura-9B bf16.
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "vllm"], check=True)
try:
    import torch
    gpu_count = torch.cuda.device_count()
except Exception:
    gpu_count = 0
if gpu_count < 1:
    raise RuntimeError("No CUDA GPU is visible. Enable a Kaggle GPU accelerator and rerun.")
tensor_parallel = 2 if gpu_count >= 2 else 1
print(f"Visible CUDA GPUs: {gpu_count}; vLLM tensor parallel size: {tensor_parallel}", flush=True)

vllm_log = open("/kaggle/working/vllm_server.log", "w", encoding="utf-8")
def start_vllm(model_id):
    vllm_cli = shutil.which("vllm")
    if not vllm_cli:
        raise RuntimeError("vLLM CLI was not installed successfully.")
    command = [
        vllm_cli, "serve", model_id,
        "--served-model-name", model_id,
        "--host", "127.0.0.1", "--port", "8000",
        "--tensor-parallel-size", str(tensor_parallel),
        "--gpu-memory-utilization", "0.88",
        "--max-model-len", "8192",
        "--trust-remote-code",
    ]
    print(f"Starting vLLM model {model_id}: {' '.join(command)}", flush=True)
    return subprocess.Popen(command, stdout=vllm_log, stderr=subprocess.STDOUT)

def wait_for_vllm(process, timeout_seconds):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        if process.poll() is not None:
            return False
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(5)
    return False

vllm_process = start_vllm(os.environ["OPENAI_MODEL"])
if not wait_for_vllm(vllm_process, 900):
    vllm_process.terminate()
    try:
        vllm_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        vllm_process.kill()
    print("Homura-9B did not become ready within 15 minutes; falling back to Homura-2B.", flush=True)
    os.environ["OPENAI_MODEL"] = "IndexTeam/Index-Homura-2B"
    vllm_process = start_vllm(os.environ["OPENAI_MODEL"])
    if not wait_for_vllm(vllm_process, 900):
        vllm_process.terminate()
        vllm_log.flush()
        tail = Path("/kaggle/working/vllm_server.log").read_text(encoding="utf-8", errors="replace")[-12000:]
        raise RuntimeError("Neither Homura-9B nor Homura-2B started. vLLM log tail:\n" + tail)
print(f"Local Homura server ready at {os.environ['OPENAI_BASE_URL']} using {os.environ['OPENAI_MODEL']}.", flush=True)


In [ ]:
# Task 2: Find the uploaded video in Kaggle input directory
import os
import glob
import shutil
import json

input_dir = "/kaggle/input"
video_path = None

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith(('.mp4', '.mkv', '.avi', '.mov')):
            video_path = os.path.join(root, file)
            break
    if video_path:
        break

if not video_path:
    raise FileNotFoundError("Video not found in Kaggle input directory!")
print(f"Found video at: {video_path}")

In [ ]:
# Execute Mazinger against the local Homura vLLM server.
import traceback
import subprocess
import sys

try:
    print("\U0001f680 Starting Mazinger Dubbing Pipeline...", flush=True)
    base_dir = "/kaggle/working/mazinger_output"
    os.makedirs(base_dir, exist_ok=True)
    if not os.environ.get("OPENAI_MODEL"):
        raise RuntimeError("Local Homura server was not initialized. Rerun the notebook from the first cell.")
    print(f"LLM provider: local {os.environ['OPENAI_MODEL']} via {os.environ['OPENAI_BASE_URL']}.", flush=True)
    cmd = [
        sys.executable, "-m", "mazinger", "dub", video_path,
        "--target-language", "Hindi",
        "--output-type", "video",
        "--device", "cuda",
        "--transcribe-method", "faster-whisper",
        "--openai-base-url", "http://localhost:8000/v1",
        "--openai-api-key", "EMPTY",
        "--llm-model", os.environ["OPENAI_MODEL"]
    ]
    print(f"Executing: {' '.join(cmd)}", flush=True)
    result = subprocess.run(cmd, cwd=base_dir, capture_output=True, text=True)
    print(f"STDOUT:\n{result.stdout}", flush=True)
    print(f"STDERR:\n{result.stderr}", flush=True)
    if result.returncode != 0:
        raise RuntimeError(f"Mazinger CLI failed with return code {result.returncode}")
    print("\u2705 Mazinger finished successfully.", flush=True)

    output_candidates = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower() == "dubbed.mp4":
                output_candidates.append(os.path.join(root, file))
    output_candidates = [path for path in output_candidates if os.path.getsize(path) > 0]
    output_video = output_candidates[-1] if output_candidates else None
    if output_video:
        shutil.copy(output_video, "/kaggle/working/output.mp4")
        print("Copied final video to /kaggle/working/output.mp4", flush=True)
    else:
        message = "Could not find a non-empty dubbed.mp4 from Mazinger."
        with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as error_file:
            error_file.write(message + "\n")
    report_data = {
        "Status": "Success" if output_video else "Failed",
        "Target_Language": "Hindi",
        "Output_File": os.path.basename(output_video) if output_video else "Missing",
        "WER": "Not measured", "Sync_Offset": "Not measured", "Speaker_Similarity": "Not measured"
    }
    with open("/kaggle/working/report.json", "w", encoding="utf-8") as report_file:
        json.dump(report_data, report_file)
    if not output_video:
        raise RuntimeError("Mazinger exited without a non-empty dubbed.mp4 output; inspect error_log.txt.")
except Exception:
    error_text = traceback.format_exc()
    print(f"\u274c Error occurred!\n{error_text}", flush=True)
    with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as error_file:
        error_file.write(error_text)
    with open("/kaggle/working/report.json", "w", encoding="utf-8") as report_file:
        json.dump({"Status": "Failed", "Target_Language": "Hindi", "Output_File": "Missing", "WER": "Not measured", "Sync_Offset": "Not measured", "Speaker_Similarity": "Not measured"}, report_file)
    raise
finally:
    if "vllm_process" in globals() and vllm_process.poll() is None:
        vllm_process.terminate()
        try:
            vllm_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            vllm_process.kill()
    if "vllm_log" in globals():
        vllm_log.flush()
        vllm_log.close()
